# 02 — Training Pipeline Walkthrough

End-to-end batched training using `services/forecast/app/train.py`.
Mirrors `./scripts/train_aarflingo.sh`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
FORECAST = ROOT / "services" / "forecast"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(FORECAST))

from app.train import train_epochs
from app.dataset import load_synthetic_dataset

print("samples", len(load_synthetic_dataset()))

## Train TriadNet (batched, validation split)

In [ ]:
out = ROOT / "artifacts" / "models" / "notebook" / "triad.pt"
result = train_epochs(epochs=20, batch_size=16, out_path=out)
print(json.dumps(result, indent=2))

## Loss / accuracy curves

In [ ]:
metrics_path = Path(result["metrics_path"])
history = json.loads(metrics_path.read_text())["history"]
epochs = range(1, len(history) + 1)
train_loss = [h["train_loss"] for h in history]
val_loss = [h["val_loss"] for h in history]
val_acc = [h["val_acc"] for h in history]

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(epochs, train_loss, label="train")
ax[0].plot(epochs, val_loss, label="val")
ax[0].set_xlabel("epoch")
ax[0].set_ylabel("loss")
ax[0].legend()
ax[0].set_title("TriadNet loss")
ax[1].plot(epochs, val_acc, color="green")
ax[1].set_xlabel("epoch")
ax[1].set_ylabel("intent val acc")
ax[1].set_title("Validation accuracy")
plt.tight_layout()
plt.show()

## Smoke infer on synthetic gaze features

In [ ]:
from app.infer import load_checkpoint, infer_sequence
from core.feature_spec import vectorize

load_checkpoint(out)
play_row = {n: 0.1 for n in [
    "dog_present", "bbox_cx", "bbox_cy", "bbox_w", "bbox_h", "motion",
    "velocity_x", "velocity_y", "gaze_door", "gaze_toy", "gaze_bowl",
    "gaze_center", "edge_left", "edge_right", "edge_top", "edge_bottom",
    "brightness", "contrast", "aspect_ratio", "arousal_proxy",
]}
play_row.update({"dog_present": 1.0, "gaze_toy": 0.9, "motion": 0.2})
seq = [vectorize(play_row)] * 15
pred = infer_sequence(seq)
print(pred)